In [1]:
from utils import *
import wandb
from datetime import datetime

In [2]:
def itertoolsBetter(dataIter):
    while True:
        for batch in dataIter:
            yield batch

In [3]:
def saveExperiment(imageModel, textModel, config, experimentName, start):
    latestPath = os.path.join("checkpoints", "finetune", "latest")
    if not os.path.exists(os.path.join("checkpoints", "finetune", "latest")):
        os.mkdir(latestPath)

    stamp = start.strftime("%Y-%m-%d %H-%M")
    timePath = os.path.join("checkpoints", "finetune", stamp)
    if not os.path.exists(timePath):
        os.mkdir(timePath)

    saveToPath(latestPath, imageModel, textModel, config, experimentName)
    saveToPath(timePath, imageModel, textModel, config, experimentName)


def saveToPath(path, imageModel, textModel, config, experimentName):
    if not os.path.exists(os.path.join(path, experimentName)):
        os.mkdir(os.path.join(path, experimentName))

    torch.save(imageModel.state_dict(), os.path.join(path, experimentName, "image.pt"))
    # textModel.save_pretrained(os.path.join(path, experimentName, "text"))
    torch.save(textModel.state_dict(), os.path.join(path, experimentName, "text.pt"))
    config.save(os.path.join(path, experimentName, "config.json"))

In [4]:
def getTrainableParams(model):
    return [p for p in model.parameters() if p.requires_grad]

def setFrozen(model, frozen: bool):
    for param in model.parameters():
        param.requires_grad = not frozen

def unfreezeTopBlocks(imageModel, textModel: CLIPTextEmbedder, n: int):
    """
    Unfreeze the last n transformer blocks of both models and keep both heads trainable.
    Image model: self.transformer.layers
    Text model:  self.transformer.layers
    """

    def freezeLayers(layers):
        for layer in layers[-n:]:
            for param in layer.parameters():
                param.requires_grad = True

    setFrozen(imageModel, True)
    layers = imageModel.model.transformer.layers
    freezeLayers(layers)

    for param in imageModel.head.parameters():
        param.requires_grad = True

    setFrozen(textModel, True)
    layers = textModel.model.encoder.layers
    freezeLayers(layers)

    for param in textModel.head.parameters():
        param.requires_grad = True

def getParamGroups(model, headLR, layerLR):
    """
    Split a model's trainable params into head vs transformer layer groups.
    """
    headParamIds = {id(p) for p in model.head.parameters()}

    headParams = [
        p for p in model.parameters()
        if p.requires_grad and id(p) in headParamIds
    ]
    layerParams = [
        p for p in model.parameters()
        if p.requires_grad and id(p) not in headParamIds
    ]

    return [
        {"params": headParams,  "lr": headLR},
        {"params": layerParams, "lr": layerLR},
    ]

def makeOptimizer(config, imageModel, textModel):
    layerLR = 0.01 * config.learningRate

    paramGroups = (
        getParamGroups(imageModel, config.learningRate, layerLR)
        + getParamGroups(textModel, config.learningRate, layerLR)
    )

    return torch.optim.Adam(paramGroups)

In [5]:
def trainModel(config, textModel, imageModel, dataset, experimentName, start, imageConfig):
    setFrozen(imageModel, True)
    setFrozen(textModel, True)
    for param in imageModel.head.parameters():
        param.requires_grad = True
    for param in textModel.head.parameters():
        param.requires_grad = True
    optimizer = makeOptimizer(config, imageModel, textModel)

    imageModel.to(device)
    textModel.to(device)

    queue = MoCoQueue(dim=config.sharedDim, size=config.queueSize)
    objective = EmbeddingLoss(temperature=0.1, alpha=0.0)
    criterion = Perplexity(objective)

    train, test = dataset.split(dataset, batchSize=config.batchSize)

    testIter = itertoolsBetter(test)

    testHistory = []

    total = 0

    run = None

    epochPhases = [
        (config.headOnlyEpochs, None),
        (config.phase1Epochs, 1),
        (config.phase2Epochs, 2),
        (config.fullEpochs, len(imageModel.model.transformer.layers)),
    ]

    try:
        for phaseEpochs, nBlocks in epochPhases:
            if phaseEpochs == 0:
                continue
            if nBlocks is None:
                setFrozen(imageModel, True)
                setFrozen(textModel, True)
                for param in imageModel.head.parameters():
                    param.requires_grad = True
                for param in textModel.head.parameters():
                    param.requires_grad = True
            else:
                unfreezeTopBlocks(imageModel, textModel, nBlocks)
            optimizer = makeOptimizer(config, imageModel, textModel)

            for epoch in range(phaseEpochs):
                progress = 0
                averageTrainLoss = 0
                averageTestLoss = 0
                for images, info, text, _ in train:
                    imageModel.train()
                    textModel.train()
                    optimizer.zero_grad()

                    imageOutputs = imageModel(images.to(device))
                    textOutputs = textModel(text.to(device))
                    loss = objective(imageOutputs, textOutputs, info.to(device))
                    trainPerplexity = criterion(imageOutputs, textOutputs, info.to(device), queue=queue)

                    queue.enqueue(imageOutputs, textOutputs, info.to(device))

                    trainLoss = loss["total"].detach().item()
                    averageTrainLoss = (averageTrainLoss * progress + trainLoss) / (progress + 1)

                    loss["total"].backward()
                    optimizer.step()

                    with torch.no_grad():
                        imageModel.eval()
                        textModel.eval()
                        images1, info1, text1, _ = next(testIter)
                        imageOutputs1 = imageModel(images1.to(device))
                        textOutputs1 = textModel(text1.to(device))
                        loss1 = objective(imageOutputs1, textOutputs1, info1.to(device))
                        testPerplexity = criterion(imageOutputs1, textOutputs1, info1.to(device), queue=queue)

                        testLoss = loss1["total"].detach().item()
                        averageTestLoss = (averageTestLoss * progress + testLoss) / (progress + 1)

                    if run is None:
                        run = wandb.init(entity="dylanberndt123-missouri-state-university", project="Briefcase", config=config.serialize())
                    
                    run.log({"Train Loss": trainLoss, 
                             "Train Perplexity": trainPerplexity.detach().item(),
                             "Train InfoNCE": loss["info"].detach().item(),
                             "Train SigREG": loss["sig"].detach().item(),
                             "Test Loss": testLoss,
                             "Test Perplexity": testPerplexity.detach().item(),
                             "Test InfoNCE": loss1["info"].detach().item(),
                             "Test SigREG": loss1["sig"].detach().item(),
                             "Train Recall@1": recallAtK(imageOutputs, textOutputs, k=1, families=info.to(device), queue=queue),
                             "Test Recall@1": recallAtK(imageOutputs1, textOutputs1, k=1, families=info1.to(device), queue=queue),
                             "Train Recall@5": recallAtK(imageOutputs, textOutputs, k=5, families=info.to(device), queue=queue),
                             "Test Recall@5": recallAtK(imageOutputs1, textOutputs1, k=5, families=info1.to(device), queue=queue),
                             "Train Recall@10": recallAtK(imageOutputs, textOutputs, k=10, families=info.to(device), queue=queue),
                             "Test Recall@10": recallAtK(imageOutputs1, textOutputs1, k=10, families=info1.to(device), queue=queue)
                             }, step=total)

                    progress += 1
                    total += 1

                    progressString = f"\r{epoch + 1} | {progress}/{len(train)} | {(progress / len(train)) * 100:.3f}%"

                    print(f"{progressString} |  Train Loss: {averageTrainLoss:.2f} | Test Loss: {averageTestLoss:.2f}",end="")

                print(f"\rEpoch {epoch + 1} | Train Loss: {averageTrainLoss:.2f} | Test Loss: {averageTestLoss:.2f}{' ' * 50}")

                testHistory.append(averageTestLoss)

    except KeyboardInterrupt:
        saveExperiment(imageModel, textModel, imageConfig, experimentName, start)
        return imageModel, textModel

    saveExperiment(imageModel, textModel, imageConfig, experimentName, start)
    return imageModel, textModel

In [6]:
queryConfig = Config().load(os.path.join("configs", "querying.json"))

In [7]:
sharedDim = queryConfig.sharedDim

textModelName = "openai/clip-vit-base-patch32"
textModel = CLIPTextEmbedder(textModelName, sharedDim).to(device)

# textModelName = "bert-base-uncased"
# textModel = BERTTextEmbedder(textModelName, sharedDim).to(device)

vit, imageConfig = ViT.load(os.path.join("checkpoints", "pretrain", "2026-06-03 04-37"))
imageModel = ViTEmbedder(vit, sharedDim).to(device)

dataset = CombinedQueryData(imageConfig.dataset, training=True)
dataset.method = "none"

dataset.setTokenizer(textModelName)

experimentName = "ViT" + " " + textModelName.replace("/", "-")

imageModel, textModel = trainModel(queryConfig, textModel, imageModel, dataset, experimentName, datetime.now(), imageConfig)

saveExperiment(imageModel, textModel, imageConfig, experimentName, datetime.now())

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_projection.weight                                         | UNEXPECTED |  | 
vision_model.encoder.l


Loading MyFonts images from dataset ====================


Fonts serialized: 148/3871

Fonts serialized: 216/3871

...

4 | 1197/1201 | 99.667% |  Train Loss: 4.41 | Test Loss: 5.83

4 | 1198/1201 | 99.750% |  Train Loss: 4.41 | Test Loss: 5.83

4 | 1199/1201 | 99.833% |  Train Loss: 4.41 | Test Loss: 5.83

4 | 1200/1201 | 99.917% |  Train Loss: 4.41 | Test Loss: 5.83

Epoch 4 | Train Loss: 4.41 | Test Loss: 5.83                                                  
